# Chapter 5 Computational Lab
## Random Variables

This notebook accompanies Chapter 5 of *Probability Theory with Python and AI*.

The chapter studies the structural theory of random variables before distributions and expectation: measurability, generated information, indicators, simple random variables, measurable transformations, algebraic operations, almost-sure equality, and threshold/truncation transformations.

### Learning goals

By the end of the lab you should be able to:

1. test finite measurability through level sets;
2. explain why measurability depends on $\mathcal F$ but not on the probability weights;
3. use the half-line criterion;
4. compute $\sigma(X)$ in finite models;
5. use indicator identities;
6. represent simple random variables canonically;
7. understand why simple random variables are later used as approximation building blocks;
8. apply Borel and continuous transformations;
9. verify measurability of sums, products, maxima, minima and finite sums;
10. distinguish pointwise equality from almost-sure equality;
11. analyze capped positive-part transformations;
12. audit AI-generated measurability claims.

> **Chapter boundary.** Distribution functions and named probability laws are postponed to the next chapter.


## 0. Setup

On finite sample spaces, measurability can be checked exactly by inspecting level sets.


In [ ]:
from fractions import Fraction
from itertools import combinations, product
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def powerset(iterable):
    items = tuple(iterable)
    return {
        frozenset(c)
        for r in range(len(items) + 1)
        for c in combinations(items, r)
    }


def fmt_set(A):
    if not A:
        return r"\varnothing"
    return r"\{" + ",".join(map(str, sorted(A))) + r"\}"


def fmt_family(F):
    ordered = sorted(F, key=lambda A: (len(A), tuple(sorted(A))))
    return r"\left\{" + r",\ ".join(fmt_set(A) for A in ordered) + r"\right\}"


def parse_set(text):
    text = text.strip()
    if not text:
        return frozenset()
    return frozenset(int(x.strip()) for x in text.split(",") if x.strip())


def parse_values(text):
    return [float(x.strip()) for x in text.split(",") if x.strip()]


def level_set(variable, value):
    return frozenset(w for w in variable if variable[w] == value)


def partition_from_variable(variable):
    return [
        level_set(variable, v)
        for v in sorted(set(variable.values()))
    ]


def sigma_from_partition(blocks):
    blocks = list(map(frozenset, blocks))
    result = set()
    for r in range(len(blocks) + 1):
        for chosen in combinations(blocks, r):
            result.add(
                frozenset().union(*chosen)
                if chosen
                else frozenset()
            )
    return frozenset(result)


def generated_sigma(variable):
    return sigma_from_partition(partition_from_variable(variable))


def is_measurable_finite(variable, omega, sigma_algebra):
    omega = frozenset(omega)
    if frozenset(variable) != omega:
        return False
    return all(
        level_set(variable, value) in sigma_algebra
        for value in set(variable.values())
    )


def indicator(event, omega):
    event = frozenset(event)
    return {w: int(w in event) for w in omega}


def capped_positive_part(x, d, u):
    return min(max(x - d, 0), u)


def equality_probability(X, Y, mass):
    return sum(
        (mass[w] for w in mass if X[w] == Y[w]),
        Fraction(0, 1),
    )


def show_result(title, *lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Finite measurability tools are ready."
    "</div>"
))


## 1. From outcomes to numerical values

A map

$$
X:\Omega\to\mathbb R
$$

is a real-valued random variable when

$$
X^{-1}(B)\in\mathcal F
$$

for every Borel set $B\in\mathcal B(\mathbb R)$.

Equivalently,

$$
X:(\Omega,\mathcal F)\to(\mathbb R,\mathcal B(\mathbb R))
$$

is measurable.


### The measure versus the $\sigma$-algebra

Whether $X$ is a random variable depends on the measurable spaces, not on the probability weights.

Changing $P$ on the same $(\Omega,\mathcal F)$ preserves measurability. Shrinking $\mathcal F$ may destroy it.


In [ ]:
omega6 = frozenset(range(1, 7))
full_sigma6 = frozenset(powerset(omega6))
coarse_sigma6 = frozenset({
    frozenset(),
    omega6,
    frozenset({1, 2, 3}),
    frozenset({4, 5, 6}),
})
X_exact = {w: w for w in omega6}

display(Markdown(
    "Exact-value map measurable with the full power set: "
    f"**{is_measurable_finite(X_exact, omega6, full_sigma6)}**"
))
display(Markdown(
    "Exact-value map measurable with the coarse sigma-algebra: "
    f"**{is_measurable_finite(X_exact, omega6, coarse_sigma6)}**"
))
display(Math(r"\{X\le1\}=\{1\}"))
display(Markdown("The singleton `{1}` is not in the coarse sigma-algebra."))


### Interactive finite measurability checker

Use the coarse die information

$$
\mathcal G=
\{\varnothing,\Omega,\{1,2,3\},\{4,5,6\}\}.
$$

Enter $X(1),\ldots,X(6)$. A finite-valued map is measurable exactly when every level set belongs to $\mathcal G$.


In [ ]:
meas_values = widgets.Text(
    value="0,0,0,1,1,1",
    description="X values",
    layout=widgets.Layout(width="650px"),
)
meas_output = widgets.Output()

def update_measurability(*_):
    with meas_output:
        clear_output(wait=True)
        try:
            values = parse_values(meas_values.value)
        except ValueError:
            display(Markdown("**Use comma-separated numerical values.**"))
            return
        if len(values) != 6:
            display(Markdown("**Provide exactly six values.**"))
            return

        X = {i + 1: values[i] for i in range(6)}
        blocks = partition_from_variable(X)
        ok = is_measurable_finite(X, omega6, coarse_sigma6)

        display(Markdown(f"**Level sets:** `{[sorted(b) for b in blocks]}`"))
        for block in blocks:
            display(Markdown(
                f"`{sorted(block)}` measurable: **{block in coarse_sigma6}**"
            ))
        display(Markdown(f"**Random variable:** {ok}"))

meas_values.observe(update_measurability, names="value")
display(widgets.VBox([meas_values, meas_output]))
update_measurability()


## 2. Indicator functions

For $A\subseteq\Omega$,

$$
\mathbf 1_A(\omega)=
\begin{cases}
1,&\omega\in A,\\
0,&\omega\notin A.
\end{cases}
$$

The exact criterion is

$$
\boxed{
\mathbf 1_A\text{ is a random variable}
\iff
A\in\mathcal F.
}
$$


In [ ]:
A_obs = frozenset({1, 2, 3})
B_unobs = frozenset({1})

IA = indicator(A_obs, omega6)
IB = indicator(B_unobs, omega6)

display(Markdown(
    f"Indicator of `{sorted(A_obs)}` measurable: "
    f"**{is_measurable_finite(IA, omega6, coarse_sigma6)}**"
))
display(Markdown(
    f"Indicator of `{sorted(B_unobs)}` measurable: "
    f"**{is_measurable_finite(IB, omega6, coarse_sigma6)}**"
))


## 3. Information generated by a random variable

The generated $\sigma$-algebra is

$$
\sigma(X)
=
\{X^{-1}(B):B\in\mathcal B(\mathbb R)\}.
$$

It is the smallest $\sigma$-algebra that makes $X$ measurable.

For a finite-valued $X$, $\sigma(X)$ consists of all unions of its level sets.


In [ ]:
info_values = widgets.Text(
    value="0,0,1,1,2,2",
    description="X values",
    layout=widgets.Layout(width="650px"),
)
info_output = widgets.Output()

def update_sigmaX(*_):
    with info_output:
        clear_output(wait=True)
        try:
            values = parse_values(info_values.value)
        except ValueError:
            display(Markdown("**Use comma-separated numerical values.**"))
            return
        if len(values) != 6:
            display(Markdown("**Provide exactly six values.**"))
            return

        X = {i + 1: values[i] for i in range(6)}
        blocks = partition_from_variable(X)
        sigmaX = generated_sigma(X)

        display(Markdown(f"**Level-set partition:** `{[sorted(b) for b in blocks]}`"))
        display(Math(r"\sigma(X)=" + fmt_family(sigmaX)))
        display(Markdown(
            f"With {len(blocks)} distinct level sets, "
            f"the generated sigma-algebra has **{len(sigmaX)}** events."
        ))

info_values.observe(update_sigmaX, names="value")
display(widgets.VBox([info_values, info_output]))
update_sigmaX()


### Information generated by an indicator

If $A\in\mathcal F$, then

$$
\sigma(\mathbf 1_A)
=
\{\varnothing,\Omega,A,A^c\}.
$$

A pass--fail observation records exactly one binary distinction.


In [ ]:
A_pass = frozenset({4, 5, 6})
I_pass = indicator(A_pass, omega6)
display(Math(r"\sigma(\mathbf 1_A)=" + fmt_family(generated_sigma(I_pass))))


## 4. Half-line criterion

A map $X:\Omega\to\mathbb R$ is a random variable if and only if

$$
\{X\le x\}\in\mathcal F
\qquad
\text{for every }x\in\mathbb R.
$$

This is the practical measurability criterion used throughout probability.


In [ ]:
half_values = widgets.Text(
    value="0,0,0,5,5,5",
    description="X values",
    layout=widgets.Layout(width="650px"),
)
half_output = widgets.Output()

def update_halfline(*_):
    with half_output:
        clear_output(wait=True)
        try:
            values = parse_values(half_values.value)
        except ValueError:
            display(Markdown("**Use comma-separated numerical values.**"))
            return
        if len(values) != 6:
            display(Markdown("**Provide exactly six values.**"))
            return

        X = {i + 1: values[i] for i in range(6)}
        distinct = sorted(set(values))
        tests = [distinct[0] - 1] + distinct
        tests += [(a + b) / 2 for a, b in zip(distinct[:-1], distinct[1:])]
        tests = sorted(set(tests))

        all_ok = True
        for x in tests:
            event = frozenset(w for w in omega6 if X[w] <= x)
            ok = event in coarse_sigma6
            all_ok = all_ok and ok
            display(Markdown(
                f"$x={x:g}$: event `{sorted(event)}`, measurable = **{ok}**"
            ))
        display(Markdown(f"**Finite threshold test passes:** {all_ok}"))

half_values.observe(update_halfline, names="value")
display(widgets.VBox([half_values, half_output]))
update_halfline()


## 5. Standard events determined by $X$

If $X$ is a random variable, then

$$
\{X<a\},\quad
\{X>a\},\quad
\{X\ge a\},\quad
\{X=a\},\quad
\{a<X\le b\}
$$

are events.

For example,

$$
\{X<a\}
=
\bigcup_{n=1}^{\infty}\{X\le a-1/n\}.
$$


In [ ]:
die_X = {w: w for w in omega6}
a, b = 2, 5

standard_events = {
    "X<a": frozenset(w for w in omega6 if die_X[w] < a),
    "X>a": frozenset(w for w in omega6 if die_X[w] > a),
    "X>=a": frozenset(w for w in omega6 if die_X[w] >= a),
    "X=a": frozenset(w for w in omega6 if die_X[w] == a),
    "a<X<=b": frozenset(w for w in omega6 if a < die_X[w] <= b),
}
for label, event in standard_events.items():
    display(Markdown(f"**{label}:** `{sorted(event)}`"))


### Exact value versus threshold information

For

$$
I=\mathbf 1_{\{X>d\}},
$$

we have

$$
\sigma(I)\subseteq\sigma(X).
$$

The indicator retains only one threshold decision and discards most of the exact-value information.


In [ ]:
d = 3
threshold_X = {w: int(die_X[w] > d) for w in omega6}

sigma_exact = generated_sigma(die_X)
sigma_threshold = generated_sigma(threshold_X)

display(Markdown(f"$|\\sigma(X)|={len(sigma_exact)}$"))
display(Markdown(f"$|\\sigma(I)|={len(sigma_threshold)}$"))
display(Markdown(
    f"Inclusion verified: **{sigma_threshold <= sigma_exact}**"
))


## 6. Indicator identities

For events $A,B$,

$$
\mathbf 1_{A^c}=1-\mathbf 1_A,
$$

$$
\mathbf 1_{A\cap B}
=
\mathbf 1_A\mathbf 1_B,
$$

$$
\mathbf 1_{A\cup B}
=
\mathbf 1_A+\mathbf 1_B-\mathbf 1_A\mathbf 1_B,
$$

and

$$
\mathbf 1_{A\triangle B}
=
|\mathbf 1_A-\mathbf 1_B|.
$$


In [ ]:
indA_text = widgets.Text(value="2,4,6", description="A")
indB_text = widgets.Text(value="4,5,6", description="B")
ind_output = widgets.Output()

def update_indicator_identities(*_):
    with ind_output:
        clear_output(wait=True)
        try:
            A = parse_set(indA_text.value)
            B = parse_set(indB_text.value)
        except ValueError:
            display(Markdown("**Use comma-separated integers.**"))
            return
        if not A <= omega6 or not B <= omega6:
            display(Markdown("**A and B must be subsets of the die space.**"))
            return

        IA = indicator(A, omega6)
        IB = indicator(B, omega6)

        lines = [
            "| omega | 1_A | 1_B | union formula | symmetric-difference formula |",
            "|---:|---:|---:|---:|---:|",
        ]

        for w in sorted(omega6):
            union_value = IA[w] + IB[w] - IA[w] * IB[w]
            sym_value = abs(IA[w] - IB[w])

            assert union_value == int(w in A | B)
            assert sym_value == int(w in A ^ B)

            lines.append(
                f"| {w} | {IA[w]} | {IB[w]} | {union_value} | {sym_value} |"
            )

        display(Markdown("\n".join(lines)))

for control in (indA_text, indB_text):
    control.observe(update_indicator_identities, names="value")

display(widgets.VBox([
    widgets.HBox([indA_text, indB_text]),
    ind_output,
]))
update_indicator_identities()


## 7. Simple random variables

A random variable is **simple** if it takes finitely many values.

If the distinct values are $x_1,\ldots,x_m$, then

$$
X
=
\sum_{i=1}^{m}
x_i\mathbf 1_{A_i},
\qquad
A_i=\{X=x_i\}.
$$

The non-empty $A_i$ form a measurable partition of $\Omega$.


In [ ]:
simple_values = widgets.Text(
    value="0,0,1,1,2,2",
    description="X values",
    layout=widgets.Layout(width="650px"),
)
simple_output = widgets.Output()

def update_simple(*_):
    with simple_output:
        clear_output(wait=True)
        try:
            values = parse_values(simple_values.value)
        except ValueError:
            display(Markdown("**Use comma-separated numerical values.**"))
            return
        if len(values) != 6:
            display(Markdown("**Provide exactly six values.**"))
            return

        X = {i + 1: values[i] for i in range(6)}
        distinct = sorted(set(values))
        for value in distinct:
            block = level_set(X, value)
            display(Markdown(
                f"$A_{{{value:g}}}=\\{{X={value:g}\\}}={sorted(block)}$"
            ))
        display(Markdown(
            f"The variable is simple because it takes **{len(distinct)}** values."
        ))

simple_values.observe(update_simple, names="value")
display(widgets.VBox([simple_values, simple_output]))
update_simple()


### Methodological preview

Simple random variables are more than convenient finite examples.

Later, non-negative random variables will be approximated from below by increasing simple random variables. The recurring strategy is

$$
\text{indicators}
\longrightarrow
\text{simple random variables}
\longrightarrow
\text{increasing simple approximations}
\longrightarrow
\text{limit theorem}
\longrightarrow
\text{general result}.
$$

Chapter 7 makes this methodology explicit.


### Binned observation

For a non-negative random variable $Y$, define

$$
X=
\begin{cases}
0,&Y\le1,\\
1,&1<Y\le10,\\
2,&Y>10.
\end{cases}
$$

Then $X$ is simple and stores only the bin containing $Y$.


In [ ]:
raw_Y = {
    1: 0.5,
    2: 1.0,
    3: 3.0,
    4: 8.0,
    5: 12.0,
    6: 40.0,
}

def bin_value(y):
    if y <= 1:
        return 0
    if y <= 10:
        return 1
    return 2

binned_X = {w: bin_value(y) for w, y in raw_Y.items()}

display(Markdown(f"Raw values: **{raw_Y}**"))
display(Markdown(f"Binned values: **{binned_X}**"))
display(Markdown(
    f"Generated-information inclusion: "
    f"**{generated_sigma(binned_X) <= generated_sigma(raw_Y)}**"
))


## 8. Historical problem: Bernoulli's urn count

For $n$ draws with replacement from a white/black urn,

$$
\Omega=\{W,B\}^n.
$$

Let

$$
X(\omega)
=
\text{number of white draws in }\omega.
$$

Then $X$ takes values $0,\ldots,n$ and is simple. Its level sets

$$
A_k=\{\omega:X(\omega)=k\}
$$

form a partition. Observing $X$ loses the order of the colors.


In [ ]:
urn_n = widgets.IntSlider(value=3, min=1, max=7, description="n")
urn_output = widgets.Output()

def update_urn(*_):
    with urn_output:
        clear_output(wait=True)
        n = urn_n.value
        omega_urn = list(product(("W", "B"), repeat=n))
        X = {
            outcome: sum(symbol == "W" for symbol in outcome)
            for outcome in omega_urn
        }

        display(Markdown(f"Sample-space size: **{2**n}**"))
        for k in range(n + 1):
            level = [outcome for outcome in omega_urn if X[outcome] == k]
            display(Markdown(f"**A_{k}:** {level}"))

        display(Markdown(
            f"The {n+1} level sets generate **{2**(n+1)}** events."
        ))

urn_n.observe(update_urn, names="value")
display(widgets.VBox([urn_n, urn_output]))
update_urn()


## 9. Measurable transformations

If $X$ is a random variable and $g:\mathbb R\to\mathbb R$ is Borel measurable, then

$$
g(X)
$$

is a random variable.

For every Borel set $B$,

$$
(g\circ X)^{-1}(B)
=
X^{-1}\bigl(g^{-1}(B)\bigr).
$$

Every continuous real function is Borel measurable.


### A discontinuous Borel transformation

The threshold function

$$
g(x)=\mathbf 1_{(c,\infty)}(x)
$$

is Borel measurable even though it is discontinuous at $c$. Therefore

$$
g(X)=\mathbf 1_{\{X>c\}}
$$

is a random variable.


In [ ]:
threshold_c = widgets.FloatSlider(
    value=3.0, min=0.0, max=6.0, step=0.5, description="c"
)
threshold_output = widgets.Output()

def update_threshold(*_):
    with threshold_output:
        clear_output(wait=True)
        c = threshold_c.value
        I = {w: int(die_X[w] > c) for w in omega6}
        display(Markdown(f"Threshold-indicator values: **{I}**"))
        display(Math(r"\sigma(I)=" + fmt_family(generated_sigma(I))))

threshold_c.observe(update_threshold, names="value")
display(widgets.VBox([threshold_c, threshold_output]))
update_threshold()


### Continuous transformation: squaring

If $g(x)=x^2$, then

$$
\{X^2\le9\}
=
\{-3\le X\le3\}.
$$


In [ ]:
signed_omega = frozenset(range(-5, 6))
signed_X = {w: w for w in signed_omega}
X2 = {w: signed_X[w] ** 2 for w in signed_omega}

lhs = frozenset(w for w in signed_omega if X2[w] <= 9)
rhs = frozenset(w for w in signed_omega if -3 <= signed_X[w] <= 3)

display(Markdown(f"Left event: **{sorted(lhs)}**"))
display(Markdown(f"Right event: **{sorted(rhs)}**"))
display(Markdown(f"Equal: **{lhs == rhs}**"))


## 10. Positive and negative parts

Define

$$
X^+=\max\{X,0\},
\qquad
X^-=\max\{-X,0\}.
$$

Then

$$
X=X^+-X^-,
$$

and

$$
|X|=X^++X^-.
$$

All three functions $X^+$, $X^-$ and $|X|$ are random variables.


In [ ]:
parts_values = widgets.Text(value="-4,0,7", description="X values")
parts_output = widgets.Output()

def update_parts(*_):
    with parts_output:
        clear_output(wait=True)
        try:
            values = parse_values(parts_values.value)
        except ValueError:
            display(Markdown("**Use comma-separated real numbers.**"))
            return

        lines = [
            "| X | X+ | X- | abs(X) |",
            "|---:|---:|---:|---:|",
        ]
        for x in values:
            xp = max(x, 0)
            xm = max(-x, 0)
            assert abs((xp - xm) - x) < 1e-12
            assert abs((xp + xm) - abs(x)) < 1e-12
            lines.append(f"| {x:g} | {xp:g} | {xm:g} | {abs(x):g} |")

        display(Markdown("\n".join(lines)))

parts_values.observe(update_parts, names="value")
display(widgets.VBox([parts_values, parts_output]))
update_parts()


## 11. Algebra of random variables

If $X$ and $Y$ are real-valued random variables, then

$$
aX+bY,\qquad
XY,\qquad
\max\{X,Y\},\qquad
\min\{X,Y\}
$$

are random variables.

Useful identities are

$$
XY
=
\frac{(X+Y)^2-(X-Y)^2}{4},
$$

$$
\max\{X,Y\}
=
\frac{X+Y+|X-Y|}{2},
$$

and

$$
\min\{X,Y\}
=
\frac{X+Y-|X-Y|}{2}.
$$

No independence assumption is required.


In [ ]:
pair_omega = frozenset(product(range(1, 4), repeat=2))
pair_sigma = frozenset(powerset(pair_omega))

X1 = {w: w[0] for w in pair_omega}
X2 = {w: w[1] for w in pair_omega}

variables = {
    "X1+X2": {w: X1[w] + X2[w] for w in pair_omega},
    "X1*X2": {w: X1[w] * X2[w] for w in pair_omega},
    "max": {w: max(X1[w], X2[w]) for w in pair_omega},
    "min": {w: min(X1[w], X2[w]) for w in pair_omega},
}

for name, variable in variables.items():
    display(Markdown(
        f"**{name} measurable:** "
        f"{is_measurable_finite(variable, pair_omega, pair_sigma)}"
    ))


### Finite sums

If $X_1,\ldots,X_n$ are random variables, then

$$
S_n=\sum_{i=1}^{n}X_i
$$

is a random variable, and so is the sample mean

$$
\overline X_n=\frac{S_n}{n}.
$$


In [ ]:
omega3 = frozenset(product((0, 1), repeat=3))
coords = [
    {w: w[i] for w in omega3}
    for i in range(3)
]
S3 = {w: sum(X[w] for X in coords) for w in omega3}
mean3 = {w: S3[w] / 3 for w in omega3}

display(Markdown(f"Possible S3 values: **{sorted(set(S3.values()))}**"))
display(Markdown(f"Possible sample-mean values: **{sorted(set(mean3.values()))}**"))


## 12. Almost-sure equality

Random variables $X$ and $Y$ are equal almost surely when

$$
P(\{X=Y\})=1.
$$

They may still differ on a probability-zero event.

Unlike measurability, almost-sure equality depends on the chosen probability measure.


In [ ]:
omega_as = frozenset({"ordinary", "null"})

P = {
    "ordinary": Fraction(1, 1),
    "null": Fraction(0, 1),
}
Q = {
    "ordinary": Fraction(1, 2),
    "null": Fraction(1, 2),
}

X_as = {"ordinary": 0, "null": 0}
Y_as = {"ordinary": 0, "null": 1}

display(Markdown(
    f"Equal a.s. under P: **{equality_probability(X_as, Y_as, P) == 1}**"
))
display(Markdown(
    f"Equal a.s. under Q: **{equality_probability(X_as, Y_as, Q) == 1}**"
))


### Almost-sure equality is an equivalence relation

It is reflexive, symmetric and transitive.

If $X=Y$ a.s. and $Y=Z$ a.s., then the union of the two exceptional null sets is still null, so $X=Z$ a.s.


In [ ]:
omega_t = frozenset({"w0", "w1", "w2"})
mass_t = {
    "w0": Fraction(1, 1),
    "w1": Fraction(0, 1),
    "w2": Fraction(0, 1),
}

X = {"w0": 0, "w1": 1, "w2": 0}
Y = {"w0": 0, "w1": 0, "w2": 0}
Z = {"w0": 0, "w1": 0, "w2": 2}

display(Markdown(f"X=Y a.s.: **{equality_probability(X, Y, mass_t) == 1}**"))
display(Markdown(f"Y=Z a.s.: **{equality_probability(Y, Z, mass_t) == 1}**"))
display(Markdown(f"X=Z a.s.: **{equality_probability(X, Z, mass_t) == 1}**"))


### Transformations preserve almost-sure equality

If

$$
X=Y\quad\text{a.s.}
$$

and $g$ is Borel measurable, then

$$
g(X)=g(Y)\quad\text{a.s.}
$$


In [ ]:
def g(x):
    return x**2 + 1

gX = {w: g(X_as[w]) for w in omega_as}
gY = {w: g(Y_as[w]) for w in omega_as}

display(Markdown(
    f"g(X)=g(Y) a.s. under P: "
    f"**{equality_probability(gX, gY, P) == 1}**"
))


## 13. Threshold and truncation transformations

Let $d\in\mathbb R$ and $u>0$. Define

$$
Y_d=(X-d)^+,
$$

$$
L_u=X\wedge u,
$$

and

$$
T_d^u=\min\{(X-d)^+,u\}.
$$

These are continuous transformations of $X$.


### Structure of the capped positive part

For real $x$,

$$
t_d^u(x)=
\begin{cases}
0,&x\le d,\\
x-d,&d<x<d+u,\\
u,&x\ge d+u.
\end{cases}
$$

Hence

$$
\{T_d^u=0\}=\{X\le d\},
$$

$$
\{0<T_d^u<u\}=\{d<X<d+u\},
$$

$$
\{T_d^u=u\}=\{X\ge d+u\}.
$$

Also,

$$
\{T_d^u>y\}
=
\begin{cases}
\Omega,&y<0,\\
\{X>d+y\},&0\le y<u,\\
\varnothing,&y\ge u.
\end{cases}
$$


In [ ]:
cap_values = widgets.Text(
    value="0,150,350,700,1000",
    description="X values",
    layout=widgets.Layout(width="650px"),
)
cap_d = widgets.IntSlider(value=200, min=0, max=600, step=50, description="d")
cap_u = widgets.IntSlider(value=500, min=50, max=700, step=50, description="u")
cap_output = widgets.Output()

def update_cap(*_):
    with cap_output:
        clear_output(wait=True)
        try:
            values = parse_values(cap_values.value)
        except ValueError:
            display(Markdown("**Use comma-separated numbers.**"))
            return

        d = cap_d.value
        u = cap_u.value
        transformed = [capped_positive_part(x, d, u) for x in values]

        lines = [
            "| X | positive part | capped value |",
            "|---:|---:|---:|",
        ]
        for x, t in zip(values, transformed):
            lines.append(
                f"| {x:g} | {max(x-d,0):g} | {t:g} |"
            )
        display(Markdown("\n".join(lines)))

        grid = np.linspace(min(values + [d]) - 50, max(values + [d + u]) + 50, 500)
        positive = np.maximum(grid - d, 0)
        capped = np.minimum(positive, u)

        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.plot(grid, positive, label="positive part")
        ax.plot(grid, capped, label="capped positive part")
        ax.axvline(d, linestyle="--")
        ax.axvline(d + u, linestyle=":")
        ax.set_xlabel("x")
        ax.set_ylabel("transformed value")
        ax.legend()
        plt.show()

for control in (cap_values, cap_d, cap_u):
    control.observe(update_cap, names="value")

display(widgets.VBox([
    cap_values,
    widgets.HBox([cap_d, cap_u]),
    cap_output,
]))
update_cap()


### Tail-event verification

For $0\le y<u$,

$$
\{T_d^u>y\}=\{X>d+y\}.
$$


In [ ]:
finite_X = {
    0: 0,
    150: 150,
    350: 350,
    700: 700,
    1000: 1000,
}
finite_space = frozenset(finite_X)
d, u = 200, 500
T = {w: capped_positive_part(finite_X[w], d, u) for w in finite_space}

for y in (-1, 0, 100, 499, 500, 600):
    lhs = frozenset(w for w in finite_space if T[w] > y)

    if y < 0:
        rhs = finite_space
    elif y < u:
        rhs = frozenset(w for w in finite_space if finite_X[w] > d + y)
    else:
        rhs = frozenset()

    display(Markdown(
        f"$y={y}$: left `{sorted(lhs)}`, right `{sorted(rhs)}`, "
        f"equal = **{lhs == rhs}**"
    ))


### Affine splitting

For $q\in[0,1]$,

$$
X=qX+(1-q)X.
$$

Both pieces are random variables because scalar multiplication is continuous.


In [ ]:
split_q = widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.05, description="q")
split_x = widgets.FloatSlider(value=100.0, min=-500.0, max=500.0, step=10.0, description="x")
split_output = widgets.Output()

def update_split(*_):
    with split_output:
        clear_output(wait=True)
        q = split_q.value
        x = split_x.value
        rhs = q * x + (1 - q) * x
        display(Markdown(
            f"$x={x:g}$, $qx+(1-q)x={rhs:g}$, equal = **{abs(rhs-x)<1e-12}**"
        ))

for control in (split_q, split_x):
    control.observe(update_split, names="value")

display(widgets.VBox([
    widgets.HBox([split_q, split_x]),
    split_output,
]))
update_split()


## 14. Coarse computational model

Let

$$
\Omega=\{0,150,350,700,1000\},
$$

and

$$
\mathcal F=
\{\varnothing,\Omega,\{0,150\},\{350,700,1000\}\}.
$$

The exact-value map is not measurable, while the threshold indicator $\mathbf 1_{\{X>200\}}$ is measurable.


In [ ]:
omega5 = frozenset({0, 150, 350, 700, 1000})
low5 = frozenset({0, 150})
high5 = frozenset({350, 700, 1000})
sigma5 = frozenset({
    frozenset(),
    omega5,
    low5,
    high5,
})

exact5 = {x: x for x in omega5}
threshold5 = {x: int(x > 200) for x in omega5}

display(Markdown(
    f"Exact-value map measurable: "
    f"**{is_measurable_finite(exact5, omega5, sigma5)}**"
))
display(Markdown(
    f"Threshold indicator measurable: "
    f"**{is_measurable_finite(threshold5, omega5, sigma5)}**"
))
display(Math(r"\sigma(\mathbf 1_{\{X>200\}})=" + fmt_family(generated_sigma(threshold5))))


### A transformation theorem has hypotheses

The theorem

$$
X\text{ measurable and }g\text{ Borel measurable}
\Longrightarrow
g(X)\text{ measurable}
$$

requires $X$ to be measurable.

Applying a continuous transformation to a non-measurable map does not automatically fix the problem.


In [ ]:
d, u = 200, 500

transformed_exact5 = {
    x: capped_positive_part(exact5[x], d, u)
    for x in omega5
}

coarse5 = {
    x: 0 if x <= 150 else 700
    for x in omega5
}
transformed_coarse5 = {
    x: capped_positive_part(coarse5[x], d, u)
    for x in omega5
}

display(Markdown(
    f"Capped transform of exact-value map measurable: "
    f"**{is_measurable_finite(transformed_exact5, omega5, sigma5)}**"
))
display(Markdown(
    f"Coarse map measurable: **{is_measurable_finite(coarse5, omega5, sigma5)}**"
))
display(Markdown(
    f"Capped transform of coarse map measurable: "
    f"**{is_measurable_finite(transformed_coarse5, omega5, sigma5)}**"
))


## 15. Guided exercise generator


In [ ]:
rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Indicator", "indicator"),
        ("Generated information", "sigma"),
        ("Simple variable", "simple"),
        ("Transformation", "transform"),
        ("Almost-sure equality", "as"),
        ("Capped transform", "cap"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_out = widgets.Output()
feedback_out = widgets.Output()
state = {}

def new_exercise(_=None):
    kind = exercise_kind.value
    if kind == "random":
        kind = rng.choice(["indicator", "sigma", "simple", "transform", "as", "cap"])

    if kind == "indicator":
        answer = "no"
        prompt = "On F={empty,Omega,{1,2},{3,4}}, is the indicator of {1} measurable? yes/no"
        hint = "An indicator is measurable iff its underlying set is an event."
        solution = r"\{1\}\notin\mathcal F,\text{ so no.}"

    elif kind == "sigma":
        answer = "4"
        prompt = "How many events are in sigma(1_A) for a nontrivial event A?"
        hint = "List empty, Omega, A and A-complement."
        solution = r"|\sigma(\mathbf 1_A)|=4."

    elif kind == "simple":
        answer = "yes"
        prompt = "A function takes three values on three measurable partition blocks. Is it simple and measurable? yes/no"
        hint = "Use the characterization theorem."
        solution = r"\text{Yes.}"

    elif kind == "transform":
        answer = "yes"
        prompt = "If X is a random variable, is X^2 a random variable? yes/no"
        hint = "The square map is continuous."
        solution = r"\text{Yes.}"

    elif kind == "as":
        answer = "no"
        prompt = "If X=Y almost surely, must X(omega)=Y(omega) for every outcome? yes/no"
        hint = "They may differ on a null event."
        solution = r"\text{No.}"

    else:
        d, u, x = 200, 500, 900
        answer = str(capped_positive_part(x, d, u))
        prompt = "Compute min((X-200)^+,500) at X=900."
        hint = "Subtract 200, take the positive part, then cap at 500."
        solution = r"\min\{700,500\}=500."

    state.clear()
    state.update(answer=str(answer).lower(), hint=hint, solution=solution)
    answer_box.value = ""

    with prompt_out:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))
    with feedback_out:
        clear_output(wait=True)

def show_hint(_):
    with feedback_out:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))

def reveal(_):
    with feedback_out:
        clear_output(wait=True)
        display(Math(state["solution"]))

def check(_):
    with feedback_out:
        clear_output(wait=True)
        guess = answer_box.value.strip().lower().replace(" ", "")
        target = state["answer"].replace(" ", "")
        display(Markdown("**Correct.**" if guess == target else "**Not yet.**"))

new_button.on_click(new_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind, new_button]),
    prompt_out,
    widgets.HBox([answer_box, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    feedback_out,
]))
new_exercise()


## 16. AI Audit

Audit any AI-generated argument using the following questions:

1. Has $\Omega$ been specified?
2. Has $\mathcal F$ been specified?
3. Is an arbitrary function being called a random variable without a measurability check?
4. In a finite model, do all level sets belong to $\mathcal F$?
5. For an indicator, is the underlying set actually measurable?
6. Is $\sigma(X)$ being confused with the full original $\mathcal F$?
7. Is a threshold or binned observation correctly recognized as coarser information?
8. If $g(X)$ is claimed measurable, is $g$ Borel measurable?
9. Is independence being incorrectly used to justify measurability of $X+Y$?
10. Is almost-sure equality being confused with pointwise equality?
11. Is the dependence of “a.s.” on the probability measure recognized?
12. Are the threshold and cap regions of $T_d^u$ identified correctly?
13. Is the measurable-transformation theorem being applied when $X$ itself is non-measurable?

### Claims to audit

- “Every function $X:\Omega\to\mathbb R$ is automatically a random variable.”
- “If $X=Y$ almost surely, then $X(\omega)=Y(\omega)$ for every $\omega$.”
- “If $X$ is a random variable, then $g(X)$ is a random variable for every function $g:\mathbb R\to\mathbb R$.”

All three statements are false as written.


### Suggested AI audit prompts

- “Give a function that is measurable under one $\sigma$-algebra and non-measurable under a coarser one.”
- “Compute $\sigma(X)$ from the level-set partition of a finite-valued random variable.”
- “Verify $\mathbf 1_{A\triangle B}=|\mathbf 1_A-\mathbf 1_B|$ pointwise.”
- “Explain why simple random variables are methodological building blocks.”
- “Give two functions equal almost surely under one probability measure but not under another.”
- “Derive the event structure of $T_d^u$ before computing any probabilities.”


## 17. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. A real-valued random variable is:",
        ["Choose...", "any function", "an F/B(R)-measurable function", "a finite-valued function"],
        "an F/B(R)-measurable function",
        r"X^{-1}(B)\in\mathcal F\text{ for every Borel }B.",
    ),
    (
        "2. Changing P while keeping (Omega,F) fixed:",
        ["Choose...", "can destroy measurability", "does not change measurability"],
        "does not change measurability",
        r"\text{Measurability depends on the measurable spaces.}",
    ),
    (
        "3. 1_A is measurable exactly when:",
        ["Choose...", "P(A)>0", "A belongs to F", "A is finite"],
        "A belongs to F",
        r"\mathbf 1_A\text{ is measurable iff }A\in\mathcal F.",
    ),
    (
        "4. A nontrivial indicator generates:",
        ["Choose...", "2 events", "4 events", "all subsets"],
        "4 events",
        r"\sigma(\mathbf 1_A)=\{\varnothing,\Omega,A,A^c\}.",
    ),
    (
        "5. A simple random variable:",
        ["Choose...", "takes finitely many values", "must be constant", "must be nonnegative"],
        "takes finitely many values",
        r"\text{Simple means finite-valued.}",
    ),
    (
        "6. If g is continuous and X is a random variable:",
        ["Choose...", "g(X) is a random variable", "g(X) is independent of X"],
        "g(X) is a random variable",
        r"\text{Continuous functions are Borel measurable.}",
    ),
    (
        "7. Measurability of X+Y requires independence:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Measurability and independence are distinct.}",
    ),
    (
        "8. If X=Y a.s., then they:",
        ["Choose...", "agree everywhere", "may differ on a null event"],
        "may differ on a null event",
        r"P(X=Y)=1\text{ permits null-set differences.}",
    ),
    (
        "9. T_d^u equals u when:",
        ["Choose...", "X<=d", "d<X<d+u", "X>=d+u"],
        "X>=d+u",
        r"\{T_d^u=u\}=\{X\ge d+u\}.",
    ),
    (
        "10. Simple random variables later serve to:",
        ["Choose...", "approximate nonnegative random variables", "replace sigma-algebras"],
        "approximate nonnegative random variables",
        r"\text{They are finite building blocks for later approximation arguments.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="380px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:660px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()

def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)
        score = sum(
            widget.value == correct
            for widget, (_, _, correct, _) in zip(quiz_widgets, quiz_data)
        )
        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))
        for i, (widget, (_, _, correct, explanation)) in enumerate(
            zip(quiz_widgets, quiz_data), 1
        ):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(f"**{mark} Question {i}:** `{correct}`"))
            display(Math(explanation))

grade_button.on_click(grade_quiz)
display(widgets.VBox(quiz_rows + [grade_button, quiz_output]))


## 18. Automatic mathematical verification


In [ ]:
# Coarse finite model.
omega = frozenset({0, 150, 350, 700, 1000})
low = frozenset({0, 150})
high = frozenset({350, 700, 1000})
sigma_algebra = frozenset({
    frozenset(),
    omega,
    low,
    high,
})

# Indicator measurability and generated information.
I_low = indicator(low, omega)
assert is_measurable_finite(I_low, omega, sigma_algebra)
assert generated_sigma(I_low) == sigma_algebra

# Exact value not measurable.
exact = {w: w for w in omega}
assert not is_measurable_finite(exact, omega, sigma_algebra)

# Indicator identities.
A = frozenset({0, 350, 1000})
B = frozenset({350, 700})
IA = indicator(A, omega)
IB = indicator(B, omega)

for w in omega:
    assert int(w not in A) == 1 - IA[w]
    assert int(w in A & B) == IA[w] * IB[w]
    assert int(w in A | B) == IA[w] + IB[w] - IA[w] * IB[w]
    assert int(w in A ^ B) == abs(IA[w] - IB[w])

# Simple measurable variable.
simple = {
    0: 0,
    150: 0,
    350: 1,
    700: 1,
    1000: 1,
}
assert is_measurable_finite(simple, omega, sigma_algebra)

# Positive/negative parts.
for x in (-10, -1, 0, 3, 12):
    xp = max(x, 0)
    xm = max(-x, 0)
    assert x == xp - xm
    assert abs(x) == xp + xm

# Capped positive part.
d, u = 200, 500
coarse_value = {w: 0 if w <= 150 else 700 for w in omega}
T = {w: capped_positive_part(coarse_value[w], d, u) for w in omega}

assert is_measurable_finite(coarse_value, omega, sigma_algebra)
assert is_measurable_finite(T, omega, sigma_algebra)

assert frozenset(w for w in omega if T[w] == 0) == frozenset(
    w for w in omega if coarse_value[w] <= d
)
assert frozenset(w for w in omega if T[w] == u) == frozenset(
    w for w in omega if coarse_value[w] >= d + u
)

for y in (-1, 0, 100, 499, 500, 600):
    lhs = frozenset(w for w in omega if T[w] > y)
    if y < 0:
        rhs = omega
    elif y < u:
        rhs = frozenset(w for w in omega if coarse_value[w] > d + y)
    else:
        rhs = frozenset()
    assert lhs == rhs

# Almost-sure equality.
mass = {
    "ordinary": Fraction(1, 1),
    "null": Fraction(0, 1),
}
X = {"ordinary": 0, "null": 0}
Y = {"ordinary": 0, "null": 1}

assert equality_probability(X, Y, mass) == 1

gX = {w: X[w] ** 2 + 1 for w in mass}
gY = {w: Y[w] ** 2 + 1 for w in mass}
assert equality_probability(gX, gY, mass) == 1

show_result(
    "All Chapter 5 checks passed",
    r"\mathbf 1_A\text{ measurable}\Longleftrightarrow A\in\mathcal F",
    r"\sigma(\mathbf 1_A)=\{\varnothing,\Omega,A,A^c\}",
    r"\mathbf 1_{A\triangle B}=|\mathbf 1_A-\mathbf 1_B|",
    r"X=X^+-X^-",
    r"\{T_d^u>y\}=\{X>d+y\}\quad(0\le y<u)",
    r"X=Y\ \mathrm{a.s.}\Longrightarrow g(X)=g(Y)\ \mathrm{a.s.}",
)


## 19. Chapter map

| Chapter concept | Computational representation |
|---|---|
| random variable | finite level-set measurability test |
| dependence on $\mathcal F$ | exact map under fine/coarse information |
| indicator | event/measurability equivalence |
| $\sigma(X)$ | unions of level-set blocks |
| half-line criterion | threshold-event checker |
| standard events | explicit interval and equality events |
| information coarsening | exact value versus threshold indicator |
| indicator identities | pointwise truth table |
| simple random variable | finite level-set partition |
| simple-variable methodology | preview of later monotone approximation |
| Bernoulli urn count | count variable and lost ordering information |
| measurable transformations | threshold and square transforms |
| positive/negative parts | pointwise decomposition |
| algebra of random variables | sums, products, maxima, minima |
| almost-sure equality | null-set examples under different measures |
| capped positive part | piecewise-linear transformation and event identities |
| coarse finite model | exact computational verification |

> A random variable is a deterministic measurable function evaluated at an outcome that is not known in advance.
